# TriadLM — build corpus_v2 (free)

Streams Wikipedia EN + Chichewa + JW300 en-ny (all license-clean) into sharded
training data. $0: Kaggle CPU is fine (no GPU needed), free bandwidth + disk.
Needs: **Internet ON**. First pass uses 5000 EN articles (~30 min); scale up after.

In [ ]:
!pip install -q datasets huggingface_hub
!git clone https://github.com/PhillipMtalika/triadlm.git 2>/dev/null; cd triadlm && git pull 2>/dev/null; echo ok
%cd triadlm
!pwd && ls

In [ ]:
# Tokenizer: reuse the 8192 one (same Wikipedia domain). Retrain only if the domain shifts.
!ls data/checkpoints/base_50m/tokenizer/ 2>/dev/null || echo MISSING-RUN-CELL-BELOW

In [ ]:
!python -m data.build_large_corpus --tokenizer-dir data/checkpoints/base_50m/tokenizer --shard-dir data/shards/corpus_v2 --manifest data/manifests/corpus_v2.json --max-en 5000 --max-jw 100000
!cat data/manifests/corpus_v2.json

In [ ]:
# Publish shards + manifest as a free public dataset (versioned, citable).
from huggingface_hub import HfApi, create_repo
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)
me = api.whoami(token)["name"]
create_repo(f"{me}/triadlm-corpus-v2", repo_type="dataset", private=False, exist_ok=True, token=token)
api.upload_folder(folder_path="data/shards/corpus_v2", repo_id=f"{me}/triadlm-corpus-v2", repo_type="dataset", token=token)
api.upload_file(path_or_fileobj="data/manifests/corpus_v2.json", path_in_repo="corpus_v2.json",
               repo_id=f"{me}/triadlm-corpus-v2", repo_type="dataset", token=token)
api.upload_folder(folder_path="data/checkpoints/base_50m/tokenizer", path_in_repo="tokenizer",
               repo_id=f"{me}/triadlm-corpus-v2", repo_type="dataset", token=token)
print("uploaded corpus_v2")